# Lecture 22: VIO and SLAM

**Simultaneous Localization and Mapping (SLAM)** asks: given a stream of sensor measurements from a moving robot, can we simultaneously figure out *where the robot is* and *build a map of the environment*?

In this notebook, we tackle the vision-based variant using a sequence of images from a moving camera. The key insight is that this is a **factor graph optimization** problem:

- **Variables**: camera poses $T_i \in SE(3)$ and 3D landmark positions $\mathbf{p}_j \in \mathbb{R}^3$
- **Factors**: reprojection errors (each keypoint constrains a pose–landmark pair), odometry measurements (noisy relative poses between frames), and a prior to make the solution unique under rigid body transformations.

In [1]:
import time

import jax
import jax.numpy as jnp
import jaxlie
import jaxls
import numpy as onp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
import plotly.graph_objects as go
from pathlib import Path
from PIL import Image
from IPython.display import display

from load_colmap import load

In [ ]:
# Load colmap data
colmap_data = load("office/colmap/sparse/0")

all_kp = jnp.concatenate([f.keypoints for f in colmap_data.frames])
all_cam_idx = jnp.concatenate([jnp.full(len(f.keypoints), i) for i, f in enumerate(colmap_data.frames)])
all_lm_idx = jnp.concatenate([f.landmark_idx for f in colmap_data.frames])

## Data

We're working with a short video of my office, reconstructed using [COLMAP](https://colmap.github.io/), a standard Structure-from-Motion pipeline. COLMAP gives us:
- **Keypoints**: $k_i^t$, 2D feature locations detected in each frame
- **Correspondences**: which keypoints $k_i^t$ observe the same 3D landmark $\ell_i$.
- **Ground-truth poses & landmarks**: used here as a reference for comparison

Let's explore the data before setting up the optimization.

In [ ]:
img_files = sorted(Path("office/images_4").glob("*.png"))
img_widget = widgets.Image(format="png", width=480)

def _on_frame_change(change):
    img_widget.value = img_files[change["new"]].read_bytes()

play   = widgets.Play(min=0, max=len(img_files) - 1, step=1, interval=50, description="▶")
slider = widgets.IntSlider(min=0, max=len(img_files) - 1, description="Frame",
                           style={"description_width": "initial"},
                           layout=widgets.Layout(width="600px"))
widgets.jslink((play, "value"), (slider, "value"))
play.observe(_on_frame_change, names="value")

img_widget.value = img_files[0].read_bytes()
display(widgets.VBox([widgets.HBox([play, slider]), img_widget]))

## Understanding correspondences

Here we visualize the detected landmarks and their correspondences across frames. Each line connects a landmark to the camera pose that observes it. Note that this is fairly optimistic for matching quality - COLMAP has already done some geometric consistency checks to filter out bad matches. In practice, will probably have many outliers that require robust loss functions.

In [ ]:
def show_correspondences(frame_i, frame_j, max_matches=50):
    """Show two frames side by side with matching keypoints connected by lines.
    
    Correspondences are found by shared landmark IDs in the COLMAP reconstruction.
    """
    img_files = sorted(Path("office/images_4").glob("*.png"))  # alphabetical = COLMAP order
    # Keypoints are in full-res pixel coords; images_4 is 4x downsampled.
    kp_scale = 0.25

    # Find shared landmarks (correspondences)
    lm_i = dict(zip(colmap_data.frames[frame_i].landmark_idx.tolist(),
                    colmap_data.frames[frame_i].keypoints))
    lm_j = dict(zip(colmap_data.frames[frame_j].landmark_idx.tolist(),
                    colmap_data.frames[frame_j].keypoints))
    shared_ids = sorted(set(lm_i) & set(lm_j))

    n_total = len(shared_ids)
    if n_total == 0:
        print(f"No correspondences between frames {frame_i} and {frame_j}")
        return

    # Subsample for display
    rng = np.random.default_rng(42)
    if n_total > max_matches:
        shared_ids = rng.choice(shared_ids, max_matches, replace=False).tolist()

    kps_i = np.array([lm_i[lid] for lid in shared_ids]) * kp_scale
    kps_j = np.array([lm_j[lid] for lid in shared_ids]) * kp_scale

    # Load images
    img_i = np.array(Image.open(img_files[frame_i]))
    img_j = np.array(Image.open(img_files[frame_j]))

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
    axes[0].imshow(img_i)
    axes[1].imshow(img_j)
    axes[0].set_title(f"Frame {frame_i}  ({img_files[frame_i].name})", fontsize=11)
    axes[1].set_title(f"Frame {frame_j}  ({img_files[frame_j].name})", fontsize=11)
    for ax in axes:
        ax.axis("off")

    colors = plt.cm.hsv(np.linspace(0, 0.95, len(shared_ids)))
    for (xi, yi), (xj, yj), c in zip(kps_i, kps_j, colors):
        axes[0].plot(xi, yi, "o", color=c, ms=5, markeredgewidth=0.5, markeredgecolor="white")
        axes[1].plot(xj, yj, "o", color=c, ms=5, markeredgewidth=0.5, markeredgecolor="white")
        con = mpatches.ConnectionPatch(
            xyA=(xi, yi), coordsA="data", axesA=axes[0],
            xyB=(xj, yj), coordsB="data", axesB=axes[1],
            color=c, lw=0.8, alpha=0.55,
        )
        fig.add_artist(con)

    fig.suptitle(
        f"Frames {frame_i} ↔ {frame_j}:  {n_total} shared landmarks  "
        f"(showing {len(shared_ids)})",
        fontsize=12,
    )
    plt.show()


n = colmap_data.n_poses
widgets.interact(
    show_correspondences,
    frame_i=widgets.IntSlider(min=0, max=n - 1, step=10, value=0,   description="Frame i", style={"description_width": "initial"}),
    frame_j=widgets.IntSlider(min=0, max=n - 1, step=10, value=10,   description="Frame j", style={"description_width": "initial"}),
    max_matches=widgets.IntSlider(min=10, max=300, step=10, value=50, description="Max matches", style={"description_width": "initial"}),
)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=colmap_data.landmarks[:, 0], y=colmap_data.landmarks[:, 1], z=colmap_data.landmarks[:, 2],
    mode="markers", marker=dict(size=1, opacity=0.3), name="landmarks",
))
fig.add_trace(go.Scatter3d(
    x=colmap_data.poses_t[:, 0], y=colmap_data.poses_t[:, 1], z=colmap_data.poses_t[:, 2],
    mode="markers+lines", marker=dict(size=4, color="red", symbol="diamond"),
    name="cameras",
))
fig.update_layout(scene=dict(aspectmode="data"))
fig.show()

## Formulating the SLAM Problem

### Camera Projection Model

To define the reprojection cost, we need a function that projects a 3D world-frame landmark into 2D image coordinates given a camera pose:

$$\mathbf{p}_{\text{cam}} = T_i^{-1} \cdot \mathbf{p}_j \qquad \text{(world} \to \text{camera frame)}$$
$$\tilde{\mathbf{u}} = K\, \mathbf{p}_{\text{cam}} \qquad \text{(apply intrinsics)}$$
$$\mathbf{u} = \frac{\tilde{\mathbf{u}}_{1:2}}{\tilde{\mathbf{u}}_3} \;/\; [W, H] \qquad \text{(normalize to } [0,1]^2\text{)}$$

In [ ]:
def project_point(camera_pose, landmark):
    landmark_cam = camera_pose.inverse() @ landmark

    # Apply camera intrinsics
    landmark_proj = colmap_data.K @ landmark_cam

    # Clamp z to avoid division by zero
    z = jnp.where(jnp.abs(landmark_proj[2]) < 1e-8, 1e-8, landmark_proj[2])

    # Normalize to [0, 1]^2
    landmark_proj = (landmark_proj[:2] / z) / jnp.array([colmap_data.W, colmap_data.H])
    return landmark_proj

### Cost Functions

The factor graph has three types of factors:
* **Reprojection error**: for each observed keypoint, we penalize the distance between the observed 2D location and the projected 3D landmark.
* **Odometry error**: for each pair of consecutive frames, we penalize the difference between the relative pose implied by the current estimates and the noisy odometry measurement.
* **Prior**: we add a small penalty on the first camera pose to make the solution unique under global rigid body transformations.

The reprojection cost uses a **Huber loss** for robustness to mismatched keypoints: 
$$\rho(r) = \begin{cases} \frac{1}{2} r^2 & \text{if } |r| \leq \delta \\ \delta (|r| - \frac{1}{2}\delta) & \text{if } |r| > \delta \end{cases}$$

In [ ]:
class LandmarkVar(jaxls.Var[jax.Array], default_factory=lambda: jnp.zeros(3)):
    """Landmark position [x, y, z]."""


@jaxls.Cost.factory
def reprojection_cost(
    vals: jaxls.VarValues,
    landmark_var: LandmarkVar,
    pose_var: jaxls.SE3Var,
    keypoints_observed: jax.Array,
) -> jax.Array:
    """Reprojection error: observed keypoint vs. projected landmark."""
    pose = vals[pose_var]
    landmark = vals[landmark_var]

    landmark_proj = project_point(pose, landmark)
    error = landmark_proj - keypoints_observed / jnp.array([colmap_data.W, colmap_data.H])

    # Huber weighting: reduces influence of large residuals (outlier keypoints).
    # weight = 1 for |r| <= delta (quadratic), weight = delta/|r| for |r| > delta (linear).
    # stop_gradient prevents instabilities from differentiating through the weights.
    delta = 0.025
    abs_r = jnp.abs(error) + 1e-8
    weight = jax.lax.stop_gradient(jnp.where(abs_r > delta, delta / abs_r, 1.0))
    return error * jnp.sqrt(weight)


@jaxls.Cost.factory
def odometry_cost(
    vals: jaxls.VarValues,
    pose_var1: jaxls.SE3Var,
    pose_var2: jaxls.SE3Var,
    rel_pose_measured: jax.Array,
) -> jax.Array:
    """Odometry factor: penalizes deviation from measured relative pose."""
    pose1 = vals[pose_var1]
    pose2 = vals[pose_var2]
    relative_transform = pose1.inverse() @ pose2
    return rel_pose_measured - relative_transform.log()


@jaxls.Cost.factory
def prior_cost(
    vals: jaxls.VarValues,
    prior_init_pose: jaxls.SE3Var,
    prior_guess: jaxlie.SE3,
) -> jax.Array:
    """Prior factor: anchors the first pose to fix gauge freedom."""
    init_pose = vals[prior_init_pose]
    return 100 * (init_pose.inverse() @ prior_guess).log()

### Building the Factor Graph

We now instantiate the variables and costs, set up initial values, and compile the problem.

**Initial values:** Camera poses and landmarks are initialized from ground truth with added noise. In a real SLAM system, you typically initialize by triangulating landmarks from the previous frames - we skip that here for simplicity.

In [ ]:
# Optionally downsample the number of poses for speed.
num_poses = colmap_data.n_poses  # use all poses

# Select num_poses evenly spaced poses to optimize over, and filter out observations that correspond to poses outside this range.
pose_indices = jnp.linspace(0, colmap_data.n_poses-1, num_poses, dtype=int)

# Create the ground-truth camera poses to compute simulated IMU measurements.
true_poses1 = jaxlie.SE3.from_rotation_and_translation(
    rotation=jaxlie.SO3(colmap_data.poses_wxyz[pose_indices[:-1]]), 
    translation=colmap_data.poses_t[pose_indices[:-1]]
)

true_poses2 = jaxlie.SE3.from_rotation_and_translation(
    rotation=jaxlie.SO3(colmap_data.poses_wxyz[pose_indices[1:]]), 
    translation=colmap_data.poses_t[pose_indices[1:]]
) 


relative_poses = (true_poses1.inverse() @ true_poses2).log()
rel_pose_measurements = relative_poses + jax.random.normal(jax.random.PRNGKey(2), shape=(num_poses-1, 6)) * jnp.array([0.01, 0.01, 0.01, 0.0025, 0.0025, 0.0025])

pose_vars = jaxls.SE3Var(id=pose_indices)
valid_obs_mask = jnp.isin(jnp.arange(colmap_data.n_poses), pose_indices)
filtered_kps = all_kp[valid_obs_mask[all_cam_idx]]
filtered_cam_idx = all_cam_idx[valid_obs_mask[all_cam_idx]]
filtered_lm_idx = all_lm_idx[valid_obs_mask[all_cam_idx]]

# Get all unique filtered lm_idx
unique_lm_idx = jnp.unique(filtered_lm_idx)

landmark_vars = LandmarkVar(id=unique_lm_idx)

costs = [
    prior_cost(jaxls.SE3Var(id=0), prior_guess=jaxlie.SE3.from_rotation_and_translation(
        rotation=jaxlie.SO3(colmap_data.poses_wxyz[pose_indices[0]]), 
        translation=colmap_data.poses_t[pose_indices[0]]
    )),  # Prior on the initial pose
    reprojection_cost(
        LandmarkVar(id=filtered_lm_idx),
        jaxls.SE3Var(id=filtered_cam_idx),
        keypoints_observed=filtered_kps,
    ),
    odometry_cost(
        jaxls.SE3Var(id=pose_indices[:-1]),
        jaxls.SE3Var(id=pose_indices[1:]),
        rel_pose_measured=rel_pose_measurements,
    ),
]

init_poses = jaxlie.SE3.from_rotation_and_translation(
    rotation=jaxlie.SO3(colmap_data.poses_wxyz[pose_indices]), 
    translation=colmap_data.poses_t[pose_indices]
) @ jaxlie.SE3.exp(jax.random.normal(jax.random.PRNGKey(0), shape=(num_poses, 6)) * jnp.array([1.0, 1.0, 1.0, 0.1, 0.1, 0.1]))

init_landmarks = colmap_data.landmarks[unique_lm_idx] + jax.random.normal(jax.random.PRNGKey(1), shape=colmap_data.landmarks[unique_lm_idx].shape) * 0.5

initial_values = jaxls.VarValues.make(
    [pose_vars.with_value(value=init_poses),
     landmark_vars.with_value(value=init_landmarks) ]
)

problem = jaxls.LeastSquaresProblem(costs, [pose_vars, landmark_vars])
problem = problem.analyze()

## Solving

We do a step-by-step solve that snapshots pose/landmark estimates at each iteration, enabling animation in the [Viser](https://viser.studio) 3D viewer.

You can also just call `solver.solve()` to get the final solution without intermediate snapshots.

In [ ]:
# Option A: Iterative solve — saves trajectory snapshots for Viser animation

iterations = 25
values = initial_values

# Storage for camera poses/landmarks
camera_poses = jnp.zeros((iterations, colmap_data.n_poses, 7))  # Store as [w, x, y, z, tx, ty, tz]
landmark_positions = jnp.zeros((iterations + 1, colmap_data.n_landmarks, 3))

lam = 1e-1

for i in range(iterations):
    # Store values
    camera_poses = camera_poses.at[i, pose_indices].set(jnp.concatenate([values[pose_vars].rotation().wxyz, values[pose_vars].translation()], axis=-1))
    landmark_positions = landmark_positions.at[i, unique_lm_idx].set(values[landmark_vars])

    values = problem.solve(values, linear_solver="cholmod", termination=jaxls.TerminationConfig(max_iterations=1), trust_region=jaxls.TrustRegionConfig(lambda_initial=lam))

    # check if values match current - adapt lambda if so.
    if jnp.allclose(values[pose_vars].rotation().wxyz, camera_poses[i, pose_indices, :4]) and jnp.allclose(values[pose_vars].translation(), camera_poses[i, pose_indices, 4:]):
        print(f"Iteration {i}: No change in values, increasing lambda.")
        lam = min(lam * 5, 1e4)
    else:
        lam = max(lam * 0.8, 1e-5)

solution = values


camera_poses = camera_poses.at[iterations, pose_indices].set(jnp.concatenate([values[pose_vars].rotation().wxyz, values[pose_vars].translation()], axis=-1))
landmark_positions = landmark_positions.at[iterations, unique_lm_idx].set(values[landmark_vars])

## Results

Let's visualize the optimization outcome:
1. **3D comparison**: initial, optimized, and COLMAP ground-truth camera trajectories and landmark clouds
2. **Reprojection quality**: do the optimized variables explain the observed keypoints?

In [ ]:
import viser
# Only create if they don't exist
if "server" not in dir():
    server = viser.ViserServer()
else:
    del server
    server = viser.ViserServer()

cameras = []
for i in pose_indices:
    cameras.append(server.scene.add_camera_frustum(
        name=f"camera_{i}",
        fov=0.8, # reasonable value in rad
        aspect=0.5,
        color=(200, 200, 200),
    ))


camera_spline = server.scene.add_spline_catmull_rom("camera_spline", color=(200, 200, 200), segments=50, points=onp.asarray(camera_poses[0, :, 4:]))
gt_camera_spline = server.scene.add_spline_catmull_rom("gt_camera_spline", color=(255, 0, 0), segments=50, points=onp.asarray(colmap_data.poses_t))

landmarks_handle = server.scene.add_point_cloud("landmarks", onp.asarray(landmark_positions[0]), colors=(16, 125, 172), point_size=0.075)
error_lines = server.scene.add_line_segments("error_lines", points=np.stack([onp.asarray(landmark_positions[0]), onp.asarray(colmap_data.landmarks)], axis=1), colors=(255, 0, 255))

gui_timestep = server.gui.add_slider(
            "Optimizer Timestep",
            min=0,
            max=iterations - 1,
            step=1,
            initial_value=0,
            disabled=True,
        )

gui_next_frame = server.gui.add_button("Next Frame", disabled=True)
gui_prev_frame = server.gui.add_button("Prev Frame", disabled=True)
gui_playing = server.gui.add_checkbox("Optimization Playing", False)

gui_camera_frame = server.gui.add_slider(
    "Camera frame",
    min=0,
    max=num_poses - 1,
    step=1,
    initial_value=0,
)

@gui_next_frame.on_click
def _(_) -> None:
    gui_timestep.value = (gui_timestep.value + 1) % iterations

@gui_prev_frame.on_click
def _(_) -> None:
    gui_timestep.value = (gui_timestep.value - 1) % iterations

@gui_playing.on_update
def _(_) -> None:
    gui_timestep.disabled = gui_playing.value
    gui_next_frame.disabled = gui_playing.value
    gui_prev_frame.disabled = gui_playing.value




def main() -> None:
    while True:
        if gui_playing.value:
            gui_timestep.value = (gui_timestep.value + 1) % iterations

        timestep = gui_timestep.value
        with server.atomic():
            for (i, cam_idx) in enumerate(pose_indices):
                cameras[i].wxyz = onp.asarray(camera_poses[timestep, cam_idx, :4])
                cameras[i].position = onp.asarray(camera_poses[timestep, cam_idx, 4:])

            error_lines.points = np.stack([onp.asarray(landmark_positions[timestep]), onp.asarray(colmap_data.landmarks)], axis=1)
            landmarks_handle.points = onp.asarray(landmark_positions[timestep])
            camera_spline.points = onp.asarray(camera_poses[timestep, :, 4:])


        time.sleep(1 / 5)

main()

### Reprojection Quality

For a chosen frame, overlay the observed keypoints with projections from initial, optimized, and GT poses:
- **blue** — observed keypoints
- **orange** — initial projection (before optimization)
- **green** — optimized projection
- **red** — COLMAP ground-truth projection (best possible)

In [ ]:
# For first frame, visualize keypoints and their corresponding landmarks
frame_idx = 250
frame = colmap_data.frames[frame_idx]
kp = frame.keypoints
lm_idx = frame.landmark_idx
lm = colmap_data.landmarks[lm_idx]

# Project landmarks into the image
pose = jaxlie.SE3.from_rotation_and_translation(
    jaxlie.SO3(colmap_data.poses_wxyz[frame_idx]),
    colmap_data.poses_t[frame_idx]
)

projected_lm = jax.vmap(lambda lm: project_point(pose, lm))(lm)

kps_normalized = kp / jnp.array([colmap_data.W, colmap_data.H])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=kps_normalized[:, 0], y=kps_normalized[:, 1],
    mode="markers", marker=dict(size=5, color="blue"), name="keypoints",
))
fig.add_trace(go.Scatter(
    x=projected_lm[:, 0], y=projected_lm[:, 1],
    mode="markers", marker=dict(size=5, color="red"), name="projected landmarks",
))
fig.update_layout(yaxis=dict(scaleanchor="x", scaleratio=1))
fig.show()

In [ ]:
# Pick a frame that was in your optimization
opt_frame_idx = 3  # index into pose_indices, not into colmap_data.frames
colmap_frame_idx = pose_indices[opt_frame_idx]

print(colmap_frame_idx)

frame = colmap_data.frames[colmap_frame_idx]
kp = frame.keypoints
kps_normalized = kp / jnp.array([colmap_data.W, colmap_data.H])

# Solved pose and landmarks
pose_opt = solution[jaxls.SE3Var(id=colmap_frame_idx)]
lm_opt = solution[LandmarkVar(id=frame.landmark_idx)]

# Project with optimized values
projected_opt = jax.vmap(lambda lm: project_point(pose_opt, lm))(lm_opt)

# Project with initial values for comparison
pose_init = initial_values[jaxls.SE3Var(id=colmap_frame_idx)]
lm_init = initial_values[LandmarkVar(id=frame.landmark_idx)]
projected_init = jax.vmap(lambda lm: project_point(pose_init, lm))(lm_init)

# Also project with COLMAP ground truth for comparison
pose_gt = jaxlie.SE3.from_rotation_and_translation(
    jaxlie.SO3(colmap_data.poses_wxyz[colmap_frame_idx]),
    colmap_data.poses_t[colmap_frame_idx],
)
lm_gt = colmap_data.landmarks[frame.landmark_idx]
projected_gt = jax.vmap(lambda lm: project_point(pose_gt, lm))(lm_gt)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=kps_normalized[:, 0], y=kps_normalized[:, 1],
    mode="markers", marker=dict(size=5, color="blue"), name="observed",
))
fig.add_trace(go.Scatter(
    x=projected_opt[:, 0], y=projected_opt[:, 1],
    mode="markers", marker=dict(size=5, color="green"), name="optimized",
))
fig.add_trace(go.Scatter(
    x=projected_gt[:, 0], y=projected_gt[:, 1],
    mode="markers", marker=dict(size=5, color="red", opacity=0.5), name="COLMAP",
))
fig.add_trace(go.Scatter(
    x=projected_init[:, 0], y=projected_init[:, 1],
    mode="markers", marker=dict(size=5, color="orange"), name="initial",
)) 
fig.update_layout(yaxis=dict(scaleanchor="x", scaleratio=1))
fig.show()